# COMP8410 A1 - Week 3 (PuLID v1.1 vs InstantID) - Colab
Runtime: **G4 or A100-40GB** (PuLID v1.1 runs out of memory on a 24 GB L4 at 1024x1024). Secrets: `HF_TOKEN`. Drive folder `comp8410_a1/` holds `w3_refs/me.jpg` (your reference photo), `results`, `outputs`, `logs`.
Before saving: Edit > Notebook settings > tick *Omit code cell output when saving* (outputs show faces). Every cell below writes its log to Drive `logs/` so results can be read without notebook output.
Disconnect the runtime when finished (Runtime > Disconnect and delete runtime).

## Cell 1: Drive, code, packages

In [ ]:
%cd /content
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'w3_refs', 'w3_refs_candidates', 'logs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!pip -q install -r requirements.txt
!pip -q install -U transformers accelerate sentencepiece protobuf diffusers


## Cell 2: link Drive folders, log in, GPU check

In [ ]:
from google.colab import userdata
os.environ['HF_HOME'] = '/content/hf_cache'   # local disk, not Drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # reduce fragmentation
!rm -rf results outputs
!ln -s {BASE}/results results
!ln -s {BASE}/outputs outputs
!mkdir -p data/processed && ln -sfn {BASE}/w3_refs data/processed/w3_refs
!mkdir -p {BASE}/w3_refs_extra && ln -sfn {BASE}/w3_refs_extra data/processed/w3_refs_extra
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q tests


## Cell 3: make the synthetic reference faces (SDXL base 1.0, about 5 minutes)
Writes 4 candidates per synthetic identity and a contact sheet to Drive `w3_refs_candidates/`.

In [ ]:
!python pairs/w3_identity_preservation/make_synthetic_refs.py --out {BASE}/w3_refs_candidates 2>&1 | tail -5


## Cell 4: keep one candidate per synthetic identity
Seeds chosen from `contact_sheet.jpg` so all three references are frontal head-and-shoulders at a similar face size (seed 0 gave syn_a and syn_c a small, half-body face).


In [ ]:
PICK = {'syn_a': 2, 'syn_b': 1, 'syn_c': 1}   # chosen from the contact sheet: frontal, sharp, largest face
from PIL import Image
for k, s in PICK.items():
    Image.open(f'{BASE}/w3_refs_candidates/{k}_cand{s}.png').convert('RGB').save(f'{BASE}/w3_refs/{k}.jpg', quality=95)
!ls -la {BASE}/w3_refs


## Cell 5: dependencies and the two official repos (pinned to the commits I read)

In [ ]:
!pip -q install insightface facexlib timm einops ftfy opencv-python-headless torchsde 2>&1 | tail -3
!pip -q uninstall -y onnxruntime onnxruntime-gpu 2>&1 | tail -1
!pip -q install onnxruntime-gpu 2>&1 | tail -2
!python -c "import onnxruntime as ort; print(ort.__version__, ort.get_available_providers())"
!mkdir -p /content/third_party && cd /content/third_party && rm -rf PuLID InstantID && git clone -q https://github.com/ToTheBeginning/PuLID.git && git clone -q https://github.com/instantX-research/InstantID.git
!cd /content/third_party/PuLID && git checkout -q 1aa2fc7df4bf51080df39f355f9abdc1cbfefbaa && git log -1 --format='PuLID %h'
!cd /content/third_party/InstantID && git checkout -q 2145b67f9607da6234702063692330185f374486 && git log -1 --format='InstantID %h'


## Cell 6: smoke tests (1 item, 1 seed each). Run the two lines separately and check each one finishes.
Full logs are saved to Drive `logs/`.

In [ ]:
!mkdir -p {BASE}/logs
!python pairs/w3_identity_preservation/run_w3.py --model pulid-v1.1 --limit 1 --seeds 0 2>&1 | tee {BASE}/logs/w3_smoke_pulid.log | tail -30


In [ ]:
!python pairs/w3_identity_preservation/run_w3.py --model instantid --limit 1 --seeds 0 2>&1 | tee {BASE}/logs/w3_smoke_instantid.log | tail -30


## Cell 7: full run (16 items x 3 seeds x 2 models = 96 images, about 10 minutes)
Clears `w3.jsonl` first: the InstantID smoke row was timed with face analysis on CPU, so it is not comparable with the rest.


In [ ]:
!rm -f {BASE}/results/w3.jsonl
!python pairs/w3_identity_preservation/run_w3.py --model pulid-v1.1 --seeds 0 1 2 2>&1 | tee {BASE}/logs/w3_full_pulid.log | tail -5
!python pairs/w3_identity_preservation/run_w3.py --model instantid --seeds 0 1 2 2>&1 | tee {BASE}/logs/w3_full_instantid.log | tail -5
!wc -l {BASE}/results/w3.jsonl   # expect 192 rows (96 generations x 2 metrics)


## Cell 8: score the images (FaceNet is the primary identity metric, about 3 minutes)
`--calibrate` also measures what the FaceNet scale means for these faces: same person in a different photo, and different people. Without that the cosine numbers have no reference point.


In [ ]:
!pip -q install facenet-pytorch --no-deps 2>&1 | tail -2
!python pairs/w3_identity_preservation/score_w3.py --calibrate 2>&1 | tee {BASE}/logs/w3_score.log | tail -30


## Cell 9: tables, charts and the qualitative grids
Keep *Omit code cell output when saving* ticked: the grids show faces.


## Cell 8b: held-out identity check (only after `w3_refs_extra/` has your two extra photos on Drive)
The first run scored this as NaN because the folder was empty, so the FaceNet scale had no upper end. `--redo` drops those NaN rows and scores them again.


In [ ]:
!ls {BASE}/w3_refs_extra
!python pairs/w3_identity_preservation/score_w3.py --metrics id_facenet_heldout --redo --calibrate 2>&1 | tail -25


In [ ]:
!python pairs/w3_identity_preservation/report_w3.py --grid 2>&1 | tee {BASE}/logs/w3_report.log
!mkdir -p {BASE}/report_assets {BASE}/outputs/w3_figures
!cp -r report_assets/w3 {BASE}/report_assets/ && cp data/processed/w3_run/figures/*.jpg {BASE}/outputs/w3_figures/

from IPython.display import Image as IPImage, display
for f in ['w3_headline', 'w3_fairness', 'w3_cost']:
    display(IPImage(filename=f'report_assets/w3/{f}.png'))
for f in ['identities', 'scenarios']:
    display(IPImage(filename=f'data/processed/w3_run/figures/w3_qualitative_{f}.jpg', width=900))
